In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import faiss
import spacy
import re
import contractions
from textblob import TextBlob

# 1. Load the Document

In [3]:
data=open('Data1.txt').read()

## 2. Text Normalization

## Converting all the characters into lower case

In [4]:
data = data.lower()

## Removing Extra Spaces

In [5]:
data  = re.sub(r'\s{2,}','',data)

## Removing Numbers Like 1.,2.

In [6]:
data = re.sub(r'\d+\.','',data)
data

'the quick brown fox jumps over the lazy dognatural language processing nlp is a fascinating field of artificial intelligence that focuses on enabling computersto understand interpret and generate human languagein a meaningful and useful way nlp combines computational linguistics—rulebased modeling of human language—withstatistical machine learning and deep learning models to processand analyze large amounts of natural language dataone of the most important tasks in nlp is text preprocessing which involves several steps tokenizationbreaking down text into individual words phrases or sentences for example dont worry becomes do nt worrylowercasingconverting all uppercase letters to lowercase for consistency hello world → hello worldremoving special characterspunctuationeliminating unnecessary symbols hello world → hello worldstopword removalfiltering out common words like the a is and or but in on at etcstemmingreducing words to their root form running runs ran → runlemmatizationconverti

## Contractions

In [7]:
data = contractions.fix(data)


## Removing Punctuations And Special Characters

In [8]:
data = re.sub(r'[^0-9a-zA-Z\s]','',data)
data

'the quick brown fox jumps over the lazy dognatural language processing nlp is a fascinating field of artificial intelligence that focuses on enabling computersto understand interpret and generate human languagein a meaningful and useful way nlp combines computational linguisticsrulebased modeling of human languagewithstatistical machine learning and deep learning models to processand analyze large amounts of natural language dataone of the most important tasks in nlp is text preprocessing which involves several steps tokenizationbreaking down text into individual words phrases or sentences for example do not worry becomes do nt worrylowercasingconverting all uppercase letters to lowercase for consistency hello world  hello worldremoving special characterspunctuationeliminating unnecessary symbols hello world  hello worldstopword removalfiltering out common words like the a is and or but in on at etcstemmingreducing words to their root form running runs ran  runlemmatizationconverting 

## Text Blob

In [9]:
# values = TextBlob(data).correct()
# values

## Spacy Lemmatization -> Converts the word into Root form

In [10]:

nlp = spacy.load('en_core_web_sm')
tokens = nlp(data)
updated_tokens = [token.lemma_ for token in tokens if not token.is_stop]
data = ' '.join(updated_tokens).strip()  # Removing extra space and char also
data

'quick brown fox jump lazy dognatural language processing nlp fascinating field artificial intelligence focus enable computersto understand interpret generate human languagein meaningful useful way nlp combine computational linguisticsrulebase modeling human languagewithstatistical machine learning deep learning model processand analyze large amount natural language dataone important task nlp text preprocessing involve step tokenizationbreake text individual word phrase sentence example worry not worrylowercasingconverte uppercase letter lowercase consistency hello world   hello worldremove special characterspunctuationeliminating unnecessary symbol hello world   hello worldstopword removalfiltering common word like etcstemmingreduce word root form run run run   runlemmatizationconverte word dictionarycanonical form well   good run   runmodern nlp application include machine translation google translate sentiment analysis analyze emotion review namedentity recognition neridentifye peop

In [11]:
tokens.ents

(google, google, gpt4, typosextra, 12345, 20240115)

### Chunking(converting doc -> Chunks)

In [12]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap = 40
)

chunks = list(set(splitter.split_text(data)))

In [13]:
print(len(chunks))

22


##  Chunk Embeddings (Converts Chunks to Vectors)

In [ ]:
embedding_model = SentenceTransformer(
     model_name_or_path='sentence-transformers/all-miniLM-L6-V2'
    
)

chunk_embeddings = embedding_model.encode(chunks).astype('float32')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [15]:
chunk_embeddings.shape

(22, 384)

In [16]:
dimension = chunk_embeddings.shape[1]
dimension


384

In [17]:
faiss.normalize_L2(chunk_embeddings)

In [18]:
index_faiss_db = faiss.IndexFlatIP(dimension)
index_faiss_db.add(chunk_embeddings)

In [19]:
def r_search(query,k=3): 
    #faiss.normalize_L2(query)
    query_embeddings = embedding_model.encode(query).astype('float32')
    query_embeddings = query_embeddings.reshape(1,-1)
    faiss.normalize_L2(query_embeddings)
    print(query_embeddings.shape)
    distance,index= index_faiss_db.search(query_embeddings,k=k)
    R_chunks = [chunks[i] for i in index[0]]
    R_str = ' '.join(R_chunks)
    promp=f'''
            You're an helpful assistant
            Assigned Task for you : Structure my output => {R_str}
            Note : 
            1)Don't add extra contents just structure mention output.
            2)If there is mistake in output correct or else keep the original output
            with structured result.
    '''
import os
import requests

API_URL = "https://router.huggingface.co/v1/chat/completions"
headers = {
    "Authorization": f"Bearer {os.environ['HF_TOKEN']}",
}

def query(payload):
    response = requests.post(API_URL, headers=headers, json=payload)
    return response.json()

response = query({
    "messages": [
        {
            "role": "user",
            "content": "What is the capital of France?"
        }
    ],
    "model": "deepseek-ai/DeepSeek-R1:novita"
})

print(response)
user_prompt = 'Explain Machine Learning ?'
user_prompt = re.sub(r'[^0-9a-zA-Z\s]','',user_prompt)

response = r_search(user_prompt)
print(response)


{'error': 'You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.'}
(1, 384)
None
